# 1. Import Libraries

In [2]:
!pip install ollama

import pandas as pd
import ollama


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


# 2. Read Data

In [3]:
df = pd.read_csv("../outputs/dataset_preprocessed_bert.csv")

# 3. Create Prompt Functions

In [4]:
def sentiment_analysis(text, model="llama3.2:3b"):
    system_prompt = """
Anda adalah mesin klasifikasi sentimen yang deterministik untuk artikel berita.

Tentukan sentimen keseluruhan teks sebagai SATU label saja:

Positive
Neutral
Negative

ATURAN LABEL:

Positive:
- Artikel terutama membahas keuntungan, peningkatan, keberhasilan, atau hasil yang positif.

Negative:
- Artikel terutama membahas kerugian, masalah, risiko, konflik, atau dampak negatif.

Neutral:
- Artikel bersifat faktual atau informatif tanpa dominasi jelas positif atau negatif.
- Termasuk laporan yang seimbang atau campuran.

ATURAN KETAT:
- Keluarkan HANYA satu kata: Positive, Neutral, atau Negative
- Tanpa penjelasan
- Tanpa tanda baca
- Tanpa teks tambahan
- Tanpa format apa pun
- Dasarkan keputusan hanya pada nada keseluruhan artikel
- Jika tidak yakin, pilih Neutral
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        options={
            "num_ctx": 6144,
            "temperature": 0
        }
    )

    return response["message"]["content"].strip()

# 4. Run

In [5]:
sentiments = []

for i, row in df.iterrows():
    while True:
        try:
            sentiment = sentiment_analysis(row["konten_preprocessed"])

            sentiments.append(sentiment)

            print(i, sentiment)
            break

        except Exception as e:
            print(f"Error at row {i}: {e}, retrying...")

df["ollama_label"] = sentiments

0 Positive
1 Positive
2 Positive
3 Negative
4 Positive
5 Negative
6 Neutral
7 Positive
8 Positive
9 Positive
10 Positive
11 Positive
12 Neutral
13 Negative
14 Negative
15 Positive
16 Negative
17 Neutral
18 Neutral
19 Neutral
20 Negative
21 Positive
22 Negative
23 Positive
24 Neutral
25 Negative
26 Negative
27 Negative
28 Neutral
29 Negative
30 Neutral
31 Negative
32 Negative
33 Positive
34 Negative
35 Neutral
36 Negative
37 Negative
38 Negative
39 Negative
40 Neutral
41 Negative
42 Neutral
43 Positive
44 Negative
45 Negative
46 Negative
47 Neutral
48 Negative
49 Negative
50 Negative
51 Negative
52 Neutral
53 Negative
54 Negative
55 Neutral
56 Negative
57 Negative
58 Negative
59 Negative
60 Negative
61 Negative
62 Negative
63 Presiden Amerika Serikat Donald Trump telah mengumumkan bahwa negara tersebut akan menaikkan tarifnya pada impor dari Kanada dan Meksiko, yang merupakan dua negara terbesar perdagangan Amerika Serikat. Pemberlakuan ini diharapkan dapat meningkatkan pendapatan as da

# 5. Save to CSV

In [ ]:
df.to_csv("../outputs/dataset_preprocessed_bert_llama.csv")